# Historical Weather Data by ZIP Code (Meteostat) - Databricks SQL Version

This notebook retrieves daily historical weather data for US ZIP codes using the Meteostat API. It supports bulk fetching for the past 2 years and incremental updates for recent days. Data is stored at the ZIP code level with basic weather columns in Databricks tables.

## Databricks Environment

Since this notebook will run in Databricks, we don't need to explicitly create a Spark session. Databricks automatically provides the `spark` variable. We'll use the Databricks-specific APIs where appropriate.

In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import pgeocode
from meteostat import Point, Daily
from pyspark.sql.functions import current_timestamp, lit, col

# In Databricks, the 'spark' session is automatically provided
# We'll use the spark session that's already available

## Configuration
Set your parameters here. You can control the ZIP codes, date ranges, and whether to refetch bulk data.

In [ ]:
# --- CONFIGURATION FLAGS ---
ZIP_CODES = ["94105", "10001", "60601", "90210", "77002"]  # Example ZIP codes
COUNTRY = "US"
BULK_YEARS = 2  # How many years back for bulk fetch
REFETCH_BULK = False  # Set True to force refetch of bulk data
BULK_DATA_TABLE = "default.bulk_weather_by_zip"  # Databricks Delta table name with database
N_DAYS = 7  # How many recent days to fetch with incremental API

## Convert ZIP Codes to Coordinates

In [ ]:
def zip_to_point(zip_codes, country=COUNTRY):
    nomi = pgeocode.Nominatim(country)
    df = nomi.query_postal_code(zip_codes)
    df = df[['postal_code', 'latitude', 'longitude']].rename(columns={'postal_code': 'zip_code'})
    df = df.dropna(subset=['latitude', 'longitude'])
    df['zip_code'] = df['zip_code'].astype(str)
    return df

zip_df = zip_to_point(ZIP_CODES)
zip_df

## Helper function to check if a table exists

In [ ]:
def table_exists(table_name):
    """Check if a table exists in Databricks"""
    tables = spark.sql("SHOW TABLES").select("tableName").collect()
    table_names = [table.tableName for table in tables]
    return table_name in table_names

## Bulk Fetch: Past 2 Years Daily Weather by ZIP Code

In [ ]:
def fetch_bulk_weather(zip_df, years=2):
    end = datetime.now().date() - timedelta(days=1)
    start = end - timedelta(days=365*years)
    # Ensure both start and end are datetime.date objects
    records = []
    for _, row in zip_df.iterrows():
        point = Point(row['latitude'], row['longitude'])
        # Convert start and end to datetime.datetime for Meteostat compatibility
        start_dt = datetime.combine(start, datetime.min.time())
        end_dt = datetime.combine(end, datetime.min.time())
        data = Daily(point, start_dt, end_dt).fetch()
        data = data.reset_index()
        data['zip_code'] = row['zip_code']
        # Keep only basic columns
        data = data[['zip_code', 'time', 'tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wspd']]
        records.append(data)
    if records:
        result_df = pd.concat(records, ignore_index=True)
        # Add load_time column
        result_df['load_time'] = datetime.now()
        return result_df
    else:
        return pd.DataFrame()

if REFETCH_BULK or not table_exists(BULK_DATA_TABLE):
    print(f"Fetching bulk weather data for all ZIP codes...")
    bulk_df = fetch_bulk_weather(zip_df, years=BULK_YEARS)
    
    # Convert pandas DataFrame to Spark DataFrame
    spark_bulk_df = spark.createDataFrame(bulk_df)
    
    # Write to Databricks table (overwrite if enabled)
    spark_bulk_df.write.format("delta").mode("overwrite").saveAsTable(BULK_DATA_TABLE)
    print(f"Saved bulk data to table {BULK_DATA_TABLE}")
else:
    print(f"Loading cached bulk data from table {BULK_DATA_TABLE}")
    # Read from Databricks table
    spark_bulk_df = spark.table(BULK_DATA_TABLE)
    bulk_df = spark_bulk_df.toPandas()

bulk_df.head()

## Incremental Fetch: Most Recent N Days

In [ ]:
def fetch_recent_weather(zip_df, n_days=7):
    end = datetime.now().date() - timedelta(days=1)
    start = end - timedelta(days=n_days-1)
    records = []
    for _, row in zip_df.iterrows():
        point = Point(row['latitude'], row['longitude'])
        # Convert start and end to datetime.datetime for Meteostat compatibility
        start_dt = datetime.combine(start, datetime.min.time())
        end_dt = datetime.combine(end, datetime.min.time())
        data = Daily(point, start_dt, end_dt).fetch()
        data = data.reset_index()
        data['zip_code'] = row['zip_code']
        data = data[['zip_code', 'time', 'tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wspd']]
        records.append(data)
    if records:
        result_df = pd.concat(records, ignore_index=True)
        # Add load_time column
        result_df['load_time'] = datetime.now()
        return result_df
    else:
        return pd.DataFrame()

recent_df = fetch_recent_weather(zip_df, n_days=N_DAYS)
recent_df.head()

## Append New Data to Bulk Data Table

In [ ]:
bulk_df.info()

In [ ]:
recent_df.info()

In [ ]:
def append_new_data(spark_bulk_df, new_df):
    # Convert the new DataFrame to a Spark DataFrame
    spark_new_df = spark.createDataFrame(new_df)
    
    # Create a temporary view for the new data
    spark_new_df.createOrReplaceTempView("new_data")
    
    # Check if any new records already exist in the bulk table
    # This is more efficient than a full join when we're just appending new records
    spark.sql(f"""
    MERGE INTO {BULK_DATA_TABLE} target
    USING new_data source
    ON target.zip_code = source.zip_code AND target.time = source.time
    WHEN MATCHED THEN
      UPDATE SET 
        tavg = source.tavg,
        tmin = source.tmin,
        tmax = source.tmax,
        prcp = source.prcp,
        snow = source.snow,
        wspd = source.wspd,
        load_time = source.load_time
    WHEN NOT MATCHED THEN
      INSERT *
    """)
    
    # Read the updated table
    updated_df = spark.table(BULK_DATA_TABLE).toPandas()
    return updated_df

# Check if bulk table exists and has data
if table_exists(BULK_DATA_TABLE):
    spark_bulk_df = spark.table(BULK_DATA_TABLE)
    full_df = append_new_data(spark_bulk_df, recent_df)
    print(f"Appended new data. Full dataset now has {len(full_df)} rows.")
else:
    # If bulk table doesn't exist, create it with the recent data
    spark_recent_df = spark.createDataFrame(recent_df)
    spark_recent_df.write.format("delta").mode("overwrite").saveAsTable(BULK_DATA_TABLE)
    full_df = recent_df
    print(f"Created new bulk table with {len(full_df)} rows.")

full_df.tail()

## Data Quality Diagnostics

In [ ]:
# Simple diagnostics
print("Missing values by column:")
print(full_df.isnull().sum())
print("\nSample data:")
print(full_df.head())

## Save Results (Optional Export to CSV)

In [ ]:
# Optionally save the final results to CSV for external use
full_df.to_csv("/dbfs/FileStore/weather_by_zip_final.csv", index=False)
print("Saved final results to /dbfs/FileStore/weather_by_zip_final.csv")

# You can also query the table directly using Databricks SQL
spark.sql(f"SELECT * FROM {BULK_DATA_TABLE} LIMIT 5").show()

# Optimize the Delta table (optional)
spark.sql(f"OPTIMIZE {BULK_DATA_TABLE}")
print(f"Optimized the {BULK_DATA_TABLE} Delta table")

## Example: Analyze or Visualize Data

In [ ]:
# Example: Average temperature by ZIP code over the last 30 days
cutoff = pd.Timestamp(datetime.now().date() - timedelta(days=30))
recent_30 = full_df[full_df['time'] >= cutoff]
grouped = recent_30.groupby('zip_code')['tavg'].mean().reset_index()
print(grouped)

# You can also do this with Databricks SQL
cutoff_str = cutoff.strftime('%Y-%m-%d')
spark.sql(f"""
SELECT zip_code, AVG(tavg) as avg_temp
FROM {BULK_DATA_TABLE}
WHERE time >= '{cutoff_str}'
GROUP BY zip_code
""").show()

## Table Summary Information

Databricks manages Spark sessions automatically, so we don't need to explicitly close them.

In [ ]:
# No need to explicitly stop the Spark session in Databricks
# It will be managed by the Databricks environment

# Display summary information about the table
spark.sql(f"DESCRIBE EXTENDED {BULK_DATA_TABLE}").show(truncate=False)